# 02 — Nettoyage et preprocessing

**Objectif** : transformer les articles bruts en données propres et exploitables par les modèles, en traitant les problèmes repérés dans `01_exploration`.

**Étapes** :
1. Supprimer le bruit : entités et balises HTML, liens, publicités, mentions de source
2. Supprimer les articles aux catégories contradictoires, les doublons et la fuite de données
3. Preprocessing : minuscules, suppression des chiffres et de la ponctuation, stopwords, lemmatisation
4. Visualiser le vocabulaire de chaque catégorie et affiner la liste de stopwords
5. Mesurer l'impact du nettoyage
6. Exporter les données nettoyées

## 0. Installation et imports

En plus des bibliothèques de la partie 1, on utilise :
- `html` : conversion des entités HTML (`&lt;` → `<`)
- `nltk` : liste de stopwords anglais et lemmatiseur WordNet
- `wordcloud` : nuages de mots
- `Counter` : comptage des mots les plus fréquents
- `lru_cache` : mémorisation des résultats de la lemmatisation, pour accélérer le traitement des 120 000 articles

NLTK a besoin de ressources téléchargées à part : la liste des stopwords et le dictionnaire WordNet.

In [ ]:
%pip install -q datasets nltk wordcloud

In [ ]:
import re
import html
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from datasets import load_dataset

import warnings
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

In [ ]:
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True);

## 1. Chargement des données brutes

On repart du dataset d'origine pour que ce notebook puisse s'exécuter de façon autonome.

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
noms_categories = dataset["train"].features["label"].names

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
for df in (train_df, test_df):
    df["label_text"] = df["label"].map(dict(enumerate(noms_categories)))

print("Train :", train_df.shape, "| Test :", test_df.shape)

## 2. Suppression du bruit

### 2.1 Fonction de suppression du bruit

La fonction `supprimer_bruit` traite, dans l'ordre :
1. les entités numériques mal converties : `#39;` → `'` et `#36;` → `$` (en retirant aussi l'espace qui les précède : *Web #39;s* → *Web's*) ;
2. l'entité `quot;` → `"` ;
3. les entités HTML (`&lt;` → `<`, `&gt;` → `>`) avec `html.unescape` ;
4. les **blocs publicitaires** (`<p>ADVERTISEMENT…` jusqu'à la fin du texte) ;
5. les **balises HTML** restantes (`<p>`, `<a href=...>`) ;
6. les **liens** (`http://…`, `www.…`) ;
7. les **liens boursiers** de Reuters : *(MLS.N: Quote, Profile, Research)* ;
8. les **mentions de source** sous leurs deux formes : *(Reuters) Reuters -* et *NEW YORK (Reuters) -* ;
9. les `\` et les espaces multiples.

Les majuscules et la ponctuation sont **conservées** à cette étape : la colonne `text` doit rester lisible pour DistilBERT, qui a besoin de phrases naturelles.

In [ ]:
def supprimer_bruit(texte):
    texte = re.sub(r" ?&?#(\d+);", lambda m: chr(int(m.group(1))), texte) # " #39;" -> '   " #36;" -> $
    texte = re.sub(r"&?quot;", '"', texte)                                # quot; -> "
    texte = html.unescape(texte)                                         # &lt; -> <   &gt; -> >
    texte = re.sub(r"<p>\s*ADVERTISEMENT.*$", " ", texte)                # bloc publicitaire en fin d'article
    texte = re.sub(r"<[^>]*>", " ", texte)                               # balises HTML
    texte = re.sub(r"https?://\S+|www\.\S+", " ", texte)                 # liens
    texte = re.sub(r"\([^()]{0,15}Quote, Profile, Research\)", " ", texte) # lien boursier Reuters
    texte = re.sub(r"\(([^()]{1,40})\)\s*\1\s*-\s*", " ", texte)          # "(Reuters) Reuters - "
    texte = re.sub(r"\([A-Za-z][A-Za-z. ]{1,30}\)\s+-\s+", " ", texte)    # "NEW YORK (Reuters) - "
    texte = texte.replace("\\", " ")                                     # barres obliques inverses
    texte = re.sub(r"\s+", " ", texte)                                   # espaces multiples
    return texte.strip()

In [ ]:
exemples = [
    train_df["text"].iloc[0],
    train_df.loc[train_df["text"].str.contains("#39;"), "text"].iloc[0],
    train_df.loc[train_df["text"].str.contains("ADVERTISEMENT"), "text"].iloc[0],
]
for texte in exemples:
    print("AVANT :", texte[:300])
    print("APRÈS :", supprimer_bruit(texte)[:300])
    print()

**Observations** :
- Exemple 1 : la mention *(Reuters) Reuters -* et le `\` ont disparu, le texte est lisible.
- Exemple 2 : *Web #39;s* devient bien *Web's* et la mention *(Reuters) -* est retirée. Le nom de la ville (*NEW YORK*) reste, ce qui n'est pas gênant.
- Exemple 3 : tout le bloc publicitaire (balises, liens, texte de la pub IBM) a été supprimé, il ne reste que l'article.

### 2.2 Application et vérification

On applique la fonction à tous les articles, puis on vérifie qu'il ne reste plus aucun des motifs de bruit repérés dans l'exploration.

In [ ]:
train_df["text"] = train_df["text"].apply(supprimer_bruit)
test_df["text"] = test_df["text"].apply(supprimer_bruit)

motifs = {"#39;": r"#39;", "quot;": r"quot;", "&lt;": r"&lt;", "<balise>": r"<[^>]+>",
          "http": r"http", "ADVERTISEMENT": r"ADVERTISEMENT", "\\": r"\\",
          "Quote, Profile, Research": r"Quote, Profile, Research",
          "(Source) Source -": r"\(([^()]{1,40})\)\s*\1\s*-", "VILLE (Source) -": r"\([A-Za-z][A-Za-z. ]{1,30}\)\s+-\s+"}
for nom, m in motifs.items():
    print(f"{nom:<20} restants dans le train : {train_df['text'].str.contains(m, regex=True).sum()}")

**Observations** : tous les motifs de bruit ont disparu. Il reste seulement :
- **4 articles** contenant « ADVERTISEMENT » : 3 qui se terminent par ce seul mot (sans publicité derrière) et 1 où il est employé normalement (*an advertisement on the giant screen*) ;
- **2 articles** avec *Quote, Profile, Research* sous une forme inhabituelle.

Ces 6 cas sur 120 000 sont négligeables.

## 3. Suppression des doublons et de la fuite de données

On recalcule la **clé** de chaque article (lettres en minuscules uniquement) **après** suppression du bruit : deux articles qui ne différaient que par une mention de source ou une entité HTML deviennent ainsi identiques et seront détectés.

### 3.1 Articles aux catégories contradictoires

Si un même article est associé à plusieurs catégories, on ne peut pas savoir laquelle est la bonne : on supprime **toutes** ses occurrences, dans le train comme dans le test.

In [ ]:
def normaliser(texte):
    return re.sub(r"[^a-z]", "", texte.lower())

train_df["cle"] = train_df["text"].apply(normaliser)
test_df["cle"] = test_df["text"].apply(normaliser)

# Clés associées à plusieurs catégories (dans le train, le test ou entre les deux)
tout = pd.concat([train_df[["cle", "label"]], test_df[["cle", "label"]]])
cles_contradictoires = set(tout.groupby("cle")["label"].nunique().loc[lambda s: s > 1].index)

avant_train, avant_test = len(train_df), len(test_df)
train_df = train_df[~train_df["cle"].isin(cles_contradictoires)]
test_df = test_df[~test_df["cle"].isin(cles_contradictoires)]

print("Articles distincts aux catégories contradictoires :", len(cles_contradictoires))
print(f"Lignes supprimées — train : {avant_train - len(train_df)} | test : {avant_test - len(test_df)}")

**Observations** : **170 articles distincts** ont des catégories contradictoires, soit **331 lignes** supprimées du train et **16** du test. C'est plus que les 44 cas trouvés dans l'exploration, pour deux raisons : la suppression du bruit fait apparaître de nouveaux doublons, et on cherche maintenant aussi les contradictions **entre** le train et le test.

### 3.2 Doublons et quasi-doublons

On ne garde qu'une seule occurrence de chaque article dans chaque jeu.

In [ ]:
avant_train, avant_test = len(train_df), len(test_df)

train_df = train_df.drop_duplicates(subset="cle").reset_index(drop=True)
test_df = test_df.drop_duplicates(subset="cle").reset_index(drop=True)

print(f"Train : {avant_train} -> {len(train_df)} ({avant_train - len(train_df)} doublons supprimés)")
print(f"Test  : {avant_test} -> {len(test_df)} ({avant_test - len(test_df)} doublons supprimés)")

**Observations** : **1 335 doublons** supprimés du train (119 669 → 118 334) et 3 du test. Là encore, c'est plus que les 424 quasi-doublons de l'exploration : beaucoup d'articles ne différaient que par leur mention de source ou par des entités HTML.

### 3.3 Fuite de données entre train et test

Un article présent dans les deux jeux fausserait l'évaluation. On le retire **du test** et on le garde dans le train, pour ne pas réduire les données d'apprentissage.

In [ ]:
communs = test_df["cle"].isin(train_df["cle"])
print("Articles du test déjà présents dans le train :", communs.sum())

test_df = test_df[~communs].reset_index(drop=True)
print("Taille du test après suppression :", len(test_df))
print("Articles communs restants :", len(set(train_df["cle"]) & set(test_df["cle"])))

**Observations** : **156 articles** du test étaient déjà présents dans le train. Après leur suppression, le test compte **7 425 articles** et il ne reste plus aucun article commun : l'évaluation des modèles sera honnête.

### 3.4 Bilan des suppressions

On vérifie la taille finale du dataset et l'équilibre des catégories après toutes les suppressions.

In [ ]:
repartition = pd.DataFrame({
    "train": train_df["label_text"].value_counts(),
    "test": test_df["label_text"].value_counts(),
})
repartition["total"] = repartition["train"] + repartition["test"]
repartition["total_%"] = (repartition["total"] / repartition["total"].sum() * 100).round(2)
repartition.loc["TOTAL"] = repartition[["train", "test", "total"]].sum().tolist() + [100.0]

print(f"Dataset final : {len(train_df) + len(test_df):,} articles (sur 127 600 au départ)")
repartition

**Observations** :
- Le dataset final compte **125 759 articles** (118 334 en train, 7 425 en test) : **1 841 articles supprimés**, soit seulement 1,4 % des données.
- Les catégories restent **équilibrées** : de 24,6 % (Sci/Tech) à 25,2 % (Sports).

## 4. Preprocessing du texte

### 4.1 Nettoyage, stopwords et lemmatisation

La fonction `preprocesser` prépare le texte pour les modèles classiques (TF-IDF) et le LSTM :
1. **minuscules** ;
2. suppression de tout ce qui n'est pas une lettre : **chiffres, ponctuation, symboles** ;
3. suppression des **stopwords** (mots outils sans valeur informative : *the, and, of…*, 198 mots dans la liste NLTK) et des mots de moins de 3 lettres ;
4. **lemmatisation** : chaque mot est ramené à sa forme de base (*companies → company*) ;
5. second filtre des stopwords **après** la lemmatisation, pour les mots qui ne deviennent des stopwords qu'une fois lemmatisés. Le premier filtre reste nécessaire car le lemmatiseur déforme certains stopwords (*was → wa*, *has → ha*).

La lemmatisation est mise en cache (`lru_cache`) : chaque mot distinct n'est lemmatisé qu'une seule fois.

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

@lru_cache(maxsize=None)
def lemmatiser(mot):
    return lemmatizer.lemmatize(mot)

def preprocesser(texte):
    texte = texte.lower()                                      # minuscules
    texte = re.sub(r"[^a-z\s]", " ", texte)                    # garde uniquement les lettres
    mots = [m for m in texte.split() if m not in stop_words and len(m) > 2]
    mots = [lemmatiser(m) for m in mots]                       # lemmatisation
    mots = [m for m in mots if m not in stop_words]            # 2e filtre après lemmatisation
    return " ".join(mots)

print("Nombre de stopwords NLTK :", len(stop_words))
for mot in ["companies", "markets", "prices", "games", "players", "computers", "elections"]:
    print(f"  {mot:<10} -> {lemmatiser(mot)}")

**Observations** : la lemmatisation fonctionne bien sur les pluriels (*companies → company*, *prices → price*, *computers → computer*). Les variantes d'un même mot sont regroupées, ce qui réduit la taille du vocabulaire.

### 4.2 Application à tout le dataset

Le résultat est stocké dans une nouvelle colonne `text_clean`. Le texte sans bruit reste dans `text`.

In [ ]:
%%time
train_df["text_clean"] = train_df["text"].apply(preprocesser)
test_df["text_clean"] = test_df["text"].apply(preprocesser)

In [ ]:
for i in range(3):
    print("ORIGINAL :", train_df.loc[i, "text"])
    print("NETTOYÉ  :", train_df.loc[i, "text_clean"])
    print()

**Observations** :
- Grâce au cache, le preprocessing des 125 000 articles prend **moins d'une seconde**.
- Le texte nettoyé ne garde que les mots porteurs de sens : *Oil and Economy Cloud Stocks' Outlook…* → *oil economy cloud stock outlook soaring crude price…*
- Les abréviations courtes disparaissent (*St.* dans *Wall St.*), ce qui est acceptable.

⚠️ On conserve les deux versions : **DistilBERT** utilisera `text`, car il a besoin des phrases complètes pour comprendre le contexte. Les modèles classiques et le LSTM utiliseront `text_clean`.

## 5. Visualisation du vocabulaire

### 5.1 Mots les plus fréquents par catégorie

On affiche les 15 mots les plus fréquents de chaque catégorie, pour vérifier que le vocabulaire est bien propre à chacune.

In [ ]:
categories = noms_categories

def top_mots(df, categorie, n):
    mots = " ".join(df.loc[df["label_text"] == categorie, "text_clean"]).split()
    return Counter(mots).most_common(n)

def afficher_top_mots(df, titre, n=15):
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for ax, categorie in zip(axes, categories):
        top = pd.DataFrame(top_mots(df, categorie, n), columns=["mot", "frequence"])
        sns.barplot(data=top, y="mot", x="frequence", ax=ax, color="steelblue")
        ax.set_title(categorie)
        ax.set_xlabel("Fréquence")
        ax.set_ylabel("")
    fig.suptitle(titre, fontsize=14)
    plt.tight_layout()
    plt.show()

afficher_top_mots(train_df, "15 mots les plus fréquents par catégorie (train)")

**Observations** :
- Chaque catégorie a un vocabulaire caractéristique : *iraq, president, minister, killed* (World), *game, win, team, season* (Sports), *oil, company, price, stock* (Business), *microsoft, software, internet, computer* (Sci/Tech).
- Certains mots apparaissent pourtant en tête de **plusieurs** catégories : *new*, *year*, *said*, *two*. Ils n'aident pas à distinguer les catégories.

### 5.2 Stopwords spécifiques au corpus

Plutôt que de choisir ces mots « à l'œil », on les repère **automatiquement** : on garde les mots présents dans le top 30 de **chacune** des 4 catégories. On les ajoute aux stopwords, puis on refait le preprocessing.

In [ ]:
tops = [{mot for mot, _ in top_mots(train_df, c, 30)} for c in categories]
stopwords_corpus = set.intersection(*tops)
print("Mots communs au top 30 des 4 catégories :", sorted(stopwords_corpus))

In [ ]:
stop_words = stop_words | stopwords_corpus

train_df["text_clean"] = train_df["text"].apply(preprocesser)
test_df["text_clean"] = test_df["text"].apply(preprocesser)

afficher_top_mots(train_df, "15 mots les plus fréquents par catégorie (après retrait des stopwords du corpus)")

**Observations** :
- La règle isole **2 mots** : *new* et *year*. Après leur retrait, *microsoft* passe en tête de Sci/Tech et de nouveaux mots discriminants apparaissent (*palestinian, bush* en World ; *quarter, billion* en Business ; *coach* en Sports).
- *said* reste fréquent en World, Business et Sci/Tech, mais pas en Sports : il n'est pas commun aux 4 catégories, donc on le garde. La pondération **TF-IDF** utilisée à l'étape suivante réduira automatiquement le poids de ces mots très répandus.

### 5.3 Nuages de mots par catégorie

Le nuage de mots affiche les mots en taille proportionnelle à leur fréquence : on voit d'un coup d'œil le vocabulaire dominant de chaque catégorie.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, categorie in zip(axes.flatten(), categories):
    texte = " ".join(train_df.loc[train_df["label_text"] == categorie, "text_clean"])
    nuage = WordCloud(width=700, height=400, background_color="white", max_words=80,
                      colormap="viridis", random_state=42).generate(texte)
    ax.imshow(nuage, interpolation="bilinear")
    ax.set_title(categorie, fontsize=14)
    ax.axis("off")
fig.suptitle("Nuages de mots par catégorie (train)", fontsize=16)
plt.tight_layout()
plt.show()

**Observations** :
- Les 4 nuages sont **très différents**, ce qui confirme que le vocabulaire permet de séparer les catégories :
  - **World** : *iraq, prime minister, government, president bush, israel, election* ;
  - **Sports** : *team, game, win, season, victory, red sox, olympic* ;
  - **Business** : *company, oil, price, quarter, market, share, wall street* ;
  - **Sci/Tech** : *microsoft, technology, software, internet, ibm, google, apple*.
- Le contexte de l'époque (2004-2005) est très visible : guerre en Irak, élection présidentielle américaine (*john kerry*), *yasser arafat*, prix du pétrole.
- Les **jours de la semaine** (*monday, tuesday…*) et *yesterday* apparaissent dans toutes les catégories : ils sont fréquents mais peu discriminants.
- Certains mots sont partagés entre deux catégories (*company* en Business et Sci/Tech, *microsoft* aussi présent en Business) : ce sont ces recouvrements, surtout entre **Business et Sci/Tech**, qui causeront probablement le plus d'erreurs de classification.

### 5.4 Textes vides après preprocessing

Un article composé uniquement de stopwords ou de chiffres deviendrait vide après le preprocessing : on vérifie qu'il n'y en a pas.

In [ ]:
vides_train = (train_df["text_clean"].str.strip() == "").sum()
vides_test = (test_df["text_clean"].str.strip() == "").sum()
print("Textes vides après preprocessing — train :", vides_train, "| test :", vides_test)

train_df = train_df[train_df["text_clean"].str.strip() != ""].reset_index(drop=True)
test_df = test_df[test_df["text_clean"].str.strip() != ""].reset_index(drop=True)

**Observations** : aucun texte vide après le preprocessing, aucune ligne n'est à supprimer.

## 6. Impact du nettoyage

On compare le nombre de mots par article et la taille du vocabulaire entre le texte brut d'origine et le texte après preprocessing.

In [ ]:
brut = dataset["train"].to_pandas()["text"]

nb_mots_brut = brut.str.split().str.len()
nb_mots_clean = train_df["text_clean"].str.split().str.len()

vocab_brut = len(set(" ".join(brut).split()))
vocab_clean = len(set(" ".join(train_df["text_clean"]).split()))

print(f"Mots par article (médiane) : {nb_mots_brut.median():.0f} -> {nb_mots_clean.median():.0f}")
print(f"Total de mots              : {nb_mots_brut.sum():,} -> {nb_mots_clean.sum():,}")
print(f"Taille du vocabulaire      : {vocab_brut:,} -> {vocab_clean:,} mots distincts")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.arange(0, 120, 2)
sns.histplot(nb_mots_brut, bins=bins, color="lightgray", label="texte brut", ax=ax)
sns.histplot(nb_mots_clean, bins=bins, color="steelblue", label="après preprocessing", ax=ax)
ax.set_title("Nombre de mots par article avant et après preprocessing (train)")
ax.set_xlabel("Nombre de mots")
ax.set_ylabel("Nombre d'articles")
ax.legend()
plt.tight_layout()
plt.show()

**Observations** :
- Un article passe de **37 à 24 mots** en médiane : le preprocessing retire environ **38 %** des mots (4,5 millions → 2,8 millions au total).
- Le vocabulaire passe de **188 110 à 54 450 mots distincts** (**−71 %**). La réduction est forte car, dans le texte brut, *Google*, *google* et *Google's* comptaient comme trois mots différents, sans parler des chiffres et du bruit HTML.
- Un vocabulaire plus petit rend les modèles plus rapides et plus robustes : ils apprennent sur des mots qui ont un vrai sens.

## 7. Export des données nettoyées

On sauvegarde les deux jeux avec 4 colonnes : `text` (sans bruit), `text_clean` (après preprocessing), `label` et `label_text`. Les fichiers sont **compressés en gzip** (`.csv.gz`) : sans compression, le train ferait environ 48 Mo, juste sous la limite de 50 Mo au-delà de laquelle GitHub déconseille de pousser un fichier ; compressé, il ne fait que 16 Mo. pandas lit ce format directement avec `pd.read_csv`.

> Sur **Colab**, les fichiers sont créés dans `data/processed/` (panneau *Fichiers* à gauche) : il faut les télécharger ou les sauvegarder sur Google Drive avant de fermer la session.

In [ ]:
# En local : dossier data/processed du projet ; sur Colab : dossier data/processed créé dans /content
OUTPUT_DIR = Path("../data/processed") if Path("../data").exists() else Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

colonnes = ["text", "text_clean", "label", "label_text"]
train_df[colonnes].to_csv(OUTPUT_DIR / "train_clean.csv.gz", index=False, compression="gzip")
test_df[colonnes].to_csv(OUTPUT_DIR / "test_clean.csv.gz", index=False, compression="gzip")

for f in sorted(OUTPUT_DIR.glob("*.csv.gz")):
    print(f"{f.name:<20} {f.stat().st_size / 1e6:.1f} Mo")
print("\ntrain :", train_df[colonnes].shape, "| test :", test_df[colonnes].shape,
      "| total :", len(train_df) + len(test_df))

**Observations** : `train_clean.csv.gz` (**118 334 articles**, 16 Mo) et `test_clean.csv.gz` (**7 425 articles**, 1 Mo) sont exportés, soit **125 759 observations** au total.

**Bilan du preprocessing** :
- Bruit HTML, publicités, liens et mentions de source supprimés
- 1 841 articles retirés (étiquettes contradictoires, doublons, fuite de données), catégories toujours équilibrées
- Texte lemmatisé, sans chiffres, ponctuation ni stopwords (NLTK + *new* et *year*)
- Vocabulaire réduit de 71 %, très distinct d'une catégorie à l'autre

➡️ Les données sont prêtes pour l'entraînement des modèles.